In [1]:
import torch as trc
from torch.utils.data import Dataset, DataLoader
import os
import pandas as pd
import numpy as np
import random

In [2]:
device = trc.device("cuda" if trc.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from LulcDataset import LulcDataset

setDir = "Dataset/Processed Data/Training"
training = LulcDataset(dir = os.path.join(setDir,"training"))
validation = LulcDataset(dir = os.path.join(setDir,"validation"))
testing = LulcDataset(dir = os.path.join(setDir,"testing"))

batch = 32
trDs = DataLoader(training, batch_size= batch, shuffle= True)
vldDs = DataLoader(validation, batch_size= batch, shuffle = True)
tstDs =  DataLoader(testing , )

In [ ]:
import torch.optim as optim
import modelArch
import torch.nn as nn

model = modelArch.UNet()
optimizer = optim.Adam(model.parameters(), lr= 3e-4, weight_decay = 1e-3)
scheduler = trc.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)
weights = [0.0, 0.22750101248873864, 3.8071337283606366, 14.997123180193531, 4.085257960095998]#First Barren wt = 14.99....
criterion = nn.CrossEntropyLoss(weight = trc.tensor(weights, dtype =trc.float32).to(device))

In [5]:
def calculate_miou(model, loader, num_classes,device):
    model.eval()
    iou_per_class = trc.zeros(num_classes).to(device)
    with trc.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            output = model(images)
            preds = trc.argmax(output, dim=1)
            for cls in range(num_classes):
                iou=1.0
                predMask = preds==cls
                lblMask = labels==cls
                intersection = (predMask & lblMask).sum()
                union = (predMask | lblMask).sum()
                if union != 0:
                    iou = intersection/union
                else:
                    iou=1
                iou_per_class[cls] += iou
                
    
    return iou_per_class/len(loader), (iou_per_class/len(loader)).mean()

In [6]:
def calculate_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with trc.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            output = model(images)
            preds = trc.argmax(output, dim=1)
            total += labels.size(0)*128*128
            correct += (preds == labels).sum().item()
            # count correct pixels
            # count total pixels
            # hint: (preds == labels).sum()
            # hint: labels.numel() gives total elements
    
    print(correct / total)

In [13]:
model.to(device)
checkpoint =trc.load("best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])
iou_per_class, miou = calculate_miou(model, testLoader, 5, device)
calculate_accuracy(model, testLoader, device)
print(f"Class 0 (Unknown)    : {iou_per_class[0]:.4f}")
print(f"Class 1 (Vegetation) : {iou_per_class[1]:.4f}")
print(f"Class 2 (Urban)      : {iou_per_class[2]:.4f}")
print(f"Class 3 (Barren)     : {iou_per_class[3]:.4f}")
print(f"Class 4 (Water)      : {iou_per_class[4]:.4f}")
print(f"mIoU                 : {miou:.4f}")

0.8657048543294271
Class 0 (Unknown)    : 0.6333
Class 1 (Vegetation) : 0.8584
Class 2 (Urban)      : 0.4451
Class 3 (Barren)     : 0.1792
Class 4 (Water)      : 0.5437
mIoU                 : 0.5320


In [40]:
trc.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'miou': miou,
    'epoch': 40
}, "best_model.pth")
